# SSJ on a GPU — does the ratio move?

On CPU, plain Simultaneous Saturated Jacobi is **24–46× slower than LAPACK
`dsyevd`** — it needs ~55–80 gemm-equivalents against `dsyevd`'s measured
8–18. A campaign of measured improvements (block-size schedule → 15→8 sweeps,
a pure-gemm IPT endgame at the basin cliff, phase-aware mixed precision)
composed to **2.33×**, leaving the best CPU configuration at **16–18×
LAPACK**. This notebook carries that composed solver, not the plain one.

The reason to look at a GPU is that SSJ's flops are *all gemm* (with
`method="gemm"`, every operation is a matmul), while `syevd` spends most of
its time in tridiagonal reduction — half of that in memory-bound `symv`,
which GPUs execute badly. So the honest question is not "is SSJ fast?" but:

> **Does the SSJ-to-incumbent ratio shrink when moving from CPU+LAPACK to
> GPU+cuSOLVER — and by how much?**

This notebook measures exactly that. It runs the same solver on the GPU and
on the CPU in one session and reports both ratios, so the answer is a
*difference of ratios* and cannot be confounded by which machine Colab gave
you.

**Before you run anything: Runtime → Change runtime type → GPU.**

### The caveat that decides how to read every number below

Consumer GPUs are deliberately crippled in float64. A T4 or P100 runs fp64 at
**1/32** of its fp32 rate; an A100 or V100 runs it at **1/2**. Eigensolvers
are normally fp64. Cell 1 *measures* your GPU's ratio rather than assuming it,
and prints which rows of the results table are the meaningful ones for the
card you actually got.

*The solver code is a standalone port of `src/ssj/` from the
`general-eigensolver` repository (the repo is private, so Colab cannot clone
it), current through attempt #11 of OPTIMIZATION_LOG.md: block schedule, predictive
Newton–Schulz tightening, IPT hybrid with the clean-hand-off QR, and the
phases-own-schedule-segments mixed policy. Deliberate GPU-specific deviations,
marked in the source: the power iterations avoid per-iteration host syncs, the
block pass is batched, and the tie branch of the angle map runs
unconditionally rather than paying a host sync to test for ties.*

In [ ]:
# =====================================================================
#  1 - GPU probe, and the fp64/fp32 ratio that frames everything below
# =====================================================================
import shutil, subprocess, time

# A CPU-only Colab runtime does not merely have no GPU -- it has no
# nvidia-smi BINARY, so probing with subprocess.run raises FileNotFoundError
# before any "is there a GPU" check can fire. Look for the tool first.
if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU on this runtime (nvidia-smi is not installed).\n"
        "Runtime > Change runtime type > Hardware accelerator: GPU, "
        "then run this cell again.")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()
if not smi:
    raise SystemExit(
        "nvidia-smi is present but reports no GPU.\n"
        "Runtime > Change runtime type > Hardware accelerator: GPU, "
        "then run this cell again.")
print("GPU:", smi)

try:
    import cupy as cp
except Exception as e:                       # not installed, or no CUDA runtime
    raise SystemExit(f"cupy is unavailable on this runtime: {e}")
import numpy as np
print("cupy", cp.__version__, "| numpy", np.__version__)


def gemm_gflops(dtype, n=2048, reps=5):
    """Sustained matmul throughput. cp.ones avoids paying for RNG."""
    A = cp.ones((n, n), dtype=dtype)
    B = cp.ones((n, n), dtype=dtype)
    A @ B                                    # warm up: autotune + workspace
    cp.cuda.Device().synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        A @ B
    cp.cuda.Device().synchronize()
    dt = time.perf_counter() - t0
    return reps * 2.0 * n ** 3 / dt / 1e9


g32 = gemm_gflops(cp.float32)
g64 = gemm_gflops(cp.float64)
ratio = g32 / g64
print(f"\nfp32 gemm : {g32:8.0f} GFLOP/s")
print(f"fp64 gemm : {g64:8.0f} GFLOP/s")
print(f"fp32:fp64 = {ratio:.1f} : 1")

if ratio > 8:
    print("\n>> This card is fp64-crippled (consumer die). float64 results below\n"
          ">> understate what SSJ does on a datacentre GPU. The 'mixed' rows are\n"
          ">> the ones that matter here; for a clean fp64 read, use an A100/V100.")
else:
    print("\n>> This card has real fp64 throughput. The float64 rows below are\n"
          ">> the meaningful comparison.")


In [ ]:
# =====================================================================
#  2 - SSJ, backend-agnostic (same code path on numpy and cupy arrays)
# =====================================================================
import numpy as np


def _am(A):
    """Array module for A, without importing cupy unless a cupy array shows up."""
    if type(A).__module__.partition(".")[0] == "cupy":
        import cupy
        return cupy
    return np


def _rdtype(A):
    return np.float32 if A.dtype in (np.float32, np.complex64) else np.float64


def off_frob(B):
    xp = _am(B)
    return float(xp.linalg.norm(B - xp.diag(xp.diag(B)), ord="fro"))


def _spec_norm(A, iters=100):
    """||A||_2 by power iteration on Hermitian A.

    A power-iteration estimate can only come in at or below the true norm,
    which makes the effective tolerance tighter, never looser.

    GPU deviation: the normalisation uses a 0-d device array instead of a
    Python float, so the loop costs one host sync at the end rather than 100.
    """
    xp = _am(A)
    v = 1.0 + 0.01 * xp.arange(A.shape[0], dtype=_rdtype(A))
    v = v / xp.linalg.norm(v)
    est = None
    for _ in range(iters):
        w = A @ v
        est = xp.linalg.norm(w)
        v = w / xp.where(est > 0, est, 1.0)
    return float(est)


def _angles(B):
    """Saturated Jacobi generator K_ij = 1/2 arctan(2|B_ij| / (d_j - d_i)).

    B must be exactly Hermitian on entry; then K is exactly anti-Hermitian.
    Zero gap -> +-pi/4 * phase(B_ij); B_ij = 0 -> 0.
    """
    xp = _am(B)
    n = B.shape[0]
    d = xp.real(xp.diag(B))
    absB = xp.abs(B)
    gap = d[None, :] - d[:, None]            # exactly antisymmetric
    # Built in place: the naive form spends a third of a sweep on temporaries.
    with np.errstate(divide="ignore", invalid="ignore"):
        theta = absB * 2.0
        theta /= gap
        xp.arctan(theta, out=theta)
        theta *= 0.5
    theta = xp.nan_to_num(theta, nan=0.0)    # 0/0 at zero gap: no rotation
    # At a zero gap with B_ij != 0 the raw formula returns +pi/4 on BOTH (i,j)
    # and (j,i) -- each sees a +0 gap -- which would break anti-Hermiticity.
    # Orient the tie by the triangle instead. Applied unconditionally: the
    # `if tie.any()` guard of the CPU version would cost a host sync per sweep.
    tie = (gap == 0.0) & (absB > 0.0)
    one = xp.ones((n, n), dtype=theta.dtype)
    theta = xp.where(tie, (np.pi / 4.0) * (xp.triu(one, 1) - xp.tril(one, -1)),
                     theta)
    if B.dtype.kind == "c":
        nz = absB > 0
        phase = xp.where(nz, B / xp.where(nz, absB, 1.0), 0.0)
        K = theta * phase
    else:
        theta *= xp.sign(B)
        K = theta
    xp.fill_diagonal(K, 0.0)
    return K


def _orth_qr(M):
    """Orthonormal factor of M, sign-fixed so diag(R) > 0."""
    xp = _am(M)
    Q, R = xp.linalg.qr(M)
    d = xp.diag(R).copy()
    if d.dtype.kind == "c":
        a = xp.abs(d)
        ph = xp.where(a > 0, d / xp.where(a > 0, a, 1.0), 1.0)
    else:
        ph = xp.where(xp.sign(d) == 0, 1.0, xp.sign(d))
    return Q * ph


def _orth_cholqr2(M):
    """Two rounds of G = M^H M, R = chol(G), M <- M R^-1.

    Safe here because sigma(X(I+K)) stays within a small factor of 1, so G is
    well-conditioned. Every operation is a gemm or a small triangular solve,
    which is why this is a GPU candidate even though it measured ~3x slower
    than Householder QR on CPU.
    """
    xp = _am(M)
    for _ in range(2):
        R = xp.linalg.cholesky(M.conj().T @ M).conj().T
        M = M @ xp.linalg.inv(R)
    return M


def _orth_ns(Y, target, max_iter=60):
    """Newton-Schulz until ||Y^H Y - I||_F < target. Requires sigma(Y) < sqrt(3).

    Each iteration's Y^H Y doubles as the error monitor, so the stopping test
    is free. Returns (Y, gemm_count).
    """
    xp = _am(Y)
    eye = xp.eye(Y.shape[0], dtype=Y.dtype)
    gemms, prev = 0, np.inf
    for _ in range(max_iter):
        G = Y.conj().T @ Y
        gemms += 1
        dev = float(xp.linalg.norm(G - eye, ord="fro"))
        if dev < target or dev > 0.9 * prev:      # converged, or stalled at roundoff
            break
        prev = dev
        Y = Y @ (1.5 * eye - 0.5 * G)
        gemms += 1
    return Y, gemms


def _pow_norm(K, v0=None, iters=25):
    """||K||_2 for anti-Hermitian K, by power iteration on K^H K.

    Returns (estimate, v) so the dominant vector warm-starts the next sweep --
    the generator changes slowly, so a warm estimate needs only a few
    iterations. A modest underestimate is safe: it weakens the cap slightly,
    and the sqrt(3) Newton-Schulz headroom absorbs it.

    GPU deviation: sync-free inner loop, as in _spec_norm.
    """
    xp = _am(K)
    if v0 is None:
        v = 1.0 + 0.01 * xp.arange(K.shape[0], dtype=_rdtype(K))
    else:
        v, iters = v0, 8
    v = v / xp.linalg.norm(v)
    est = None
    for _ in range(iters):
        w = K.conj().T @ (K @ v)
        est = xp.linalg.norm(w)
        v = w / xp.where(est > 0, est, 1.0)
    return float(xp.sqrt(est)), xp.where(est > 0, v, 1.0)


def ssj_eigh(A, tol=1e-13, method="gemm", max_sweeps=1000, ns_switch=0.5,
             gemm_cap=1.0, gemm_ns_factor=0.05, X0=None, precision="full",
             mixed_switch=1e-4, block_m=0, block_passes=2, block_fn=None,
             return_info=False):
    """Eigendecomposition of Hermitian A by Simultaneous Saturated Jacobi.

        B = X^H A X ;  K_ij = 1/2 atan(2 B_ij / (d_j - d_i)) ;  X <- orth(X(I+K))

    method   : "gemm" (factorization-free, all matmul -- the GPU choice),
               "qr", "auto" (QR then a Newton-Schulz endgame), "cholqr2".
    precision: "mixed" runs the linear phase in fp32 down to `mixed_switch`,
               then warm-starts an fp64 phase from that basis. Sound because
               the map is memoryless: every sweep re-derives its angles from a
               fresh B, so low-precision sweeps cannot poison the final
               accuracy, only the warm start's quality.
    block_m  : >0 enables the SSJ-BC block-cluster preconditioner from cell 3.
    """
    xp = _am(A)
    A = xp.asarray(A)

    if precision == "mixed":
        low = np.complex64 if A.dtype.kind == "c" else np.float32
        full = np.complex128 if A.dtype.kind == "c" else np.float64
        _, V_low, i_low = ssj_eigh(
            A.astype(low), tol=max(mixed_switch, tol), method=method,
            max_sweeps=min(max_sweeps, 100), ns_switch=ns_switch,
            gemm_cap=gemm_cap, gemm_ns_factor=gemm_ns_factor,
            X0=None if X0 is None else xp.asarray(X0).astype(low),
            block_m=block_m, block_passes=block_passes, block_fn=block_fn,
            return_info=True)
        # Phases own schedule SEGMENTS: the fp64 phase warm-starts past the
        # spread problem, so the schedule's big head blocks would be wasted
        # fp64 eigensolves -- but its small TAIL blocks stay, because
        # structure below fp32's ~1e-7 resolution (tight clusters, ties)
        # reaches this phase unresolved and small blocks resolve it.
        block_full = block_m if np.ndim(block_m) == 0 else block_m[-1]
        w, V, info = ssj_eigh(
            A.astype(full), tol=tol, method=method, max_sweeps=max_sweeps,
            ns_switch=ns_switch, gemm_cap=gemm_cap,
            gemm_ns_factor=gemm_ns_factor, X0=V_low.astype(full),
            block_m=block_full, block_passes=block_passes, block_fn=block_fn,
            return_info=True)
        info["sweeps_low"] = i_low["sweeps"]
        info["sweeps_total"] = i_low["sweeps"] + info["sweeps"]
        info["gemms"] += i_low["gemms"]
        return (w, V, info) if return_info else (w, V)

    n = A.shape[0]
    # block_m: int, or a per-sweep schedule (sequence, last entry repeats).
    # Big blocks early inject the diagonal spread the first sweeps are
    # bottlenecked on; small blocks late do the adjacent decoupling that is
    # all the endgame needs. [n//2, n//4, 32] measured GOE n=800 at 8 sweeps
    # against fixed-32's 15 on CPU.
    if np.ndim(block_m) == 0:
        block_sched = None
        block_m = min(int(block_m), n // 2)
        block_active = bool(block_m) and block_fn is not None
    else:
        block_sched = tuple(min(int(m), n // 2) for m in block_m) or (0,)
        block_m = block_sched[0]
        block_active = any(block_sched) and block_fn is not None
    norm_A = _spec_norm(A)
    X = xp.eye(n, dtype=A.dtype) if X0 is None else _orth_qr(
        xp.asarray(X0).astype(A.dtype, copy=False))
    eye = xp.eye(n, dtype=A.dtype)
    history, gemms, sweeps, converged, pv = [], 0, 0, False, None
    prev_rel = 0.0

    for _ in range(max_sweeps):
        B = X.conj().T @ (A @ X)
        B = (B + B.conj().T) / 2.0           # exact symmetry for the angle map
        rel_off = off_frob(B) / norm_A
        history.append(rel_off)
        if rel_off <= tol:
            converged = True
            break

        if block_sched is not None:
            block_m = block_sched[min(sweeps, len(block_sched) - 1)]
        if block_m and block_fn is not None:
            for r in range(block_passes):
                B, X = block_fn(B, X, block_m,
                                0 if (sweeps + r) % 2 == 0 else block_m // 2)
            rel2 = off_frob(B) / norm_A
            if rel2 <= tol:
                converged = True
                history.append(rel2)
                sweeps += 1
                break

        K = _angles(B)
        Y = X @ (eye + K)                    # = X + X @ K

        if method == "qr":
            X = _orth_qr(Y)
        elif method in ("auto", "cholqr2"):
            if float(xp.linalg.norm(K, ord="fro")) < ns_switch:
                # Target the quadratic-tail scale rel_off^2, floored at a
                # fraction of tol: with the block pass the error can fall
                # several orders in ONE sweep, and then rel_off^2 is no longer
                # below the NEXT error, so the orthogonality defect would
                # survive into the answer.
                tgt = rel_off * rel_off
                if block_active and prev_rel > 0.0 and \
                        rel_off * (rel_off / prev_rel) <= tol * 1e9:
                    tgt = min(tgt, 0.1 * tol)   # tighten only near convergence
                X, g = _orth_ns(Y, target=tgt)
                gemms += g
            else:
                X = _orth_qr(Y) if method == "auto" else _orth_cholqr2(Y)
        elif method == "gemm":
            gemms += 3                       # B (2) + X @ K (1)
            sigma, pv = _pow_norm(K, v0=pv)
            if sigma > gemm_cap:
                K = K * (gemm_cap / sigma)
                Y = X @ (eye + K)
                gemms += 1
            # The polar factor is invariant under positive scaling, and
            # sigma(I+K) spans [1, sqrt(1+s^2)] exactly for anti-Hermitian K,
            # so dividing by the geometric mean centres the singular values
            # around 1 and saves Newton-Schulz iterations.
            s = min(sigma, gemm_cap)
            if s > 0.1:
                Y = Y / (1.0 + s * s) ** 0.25
            tgt = gemm_ns_factor * rel_off
            if block_active and prev_rel > 0.0 and \
                    rel_off * (rel_off / prev_rel) <= tol * 1e9:
                # The always-on floor here was measured as a 1.60x penalty on
                # CPU: it demanded 1e-14 orthogonality from the FIRST sweep.
                # The defect only has to be small on the LAST sweep -- the
                # product-form retraction re-measures it every sweep -- so
                # tighten only when the observed contraction predicts
                # convergence within 1e9 * tol (wide, because a block pass
                # can fall six orders in one sweep).
                tgt = min(tgt, 0.1 * tol)
            X, g = _orth_ns(Y, target=tgt)
            gemms += g
        else:
            raise ValueError(f"unknown method {method!r}")
        prev_rel = rel_off
        sweeps += 1
    else:
        B = X.conj().T @ (A @ X)
        B = (B + B.conj().T) / 2.0
        history.append(off_frob(B) / norm_A)

    w = xp.real(xp.diag(B))
    order = np.argsort(w, kind="stable") if xp is np else xp.argsort(w)
    w, V = w[order], X[:, order]
    info = {"sweeps": sweeps, "gemms": gemms, "history": history,
            "converged": converged, "norm_A": norm_A}
    return (w, V, info) if return_info else (w, V)


print("SSJ loaded.")

In [ ]:
# =====================================================================
#  3 - SSJ-BC: the block-cluster preconditioner, batched for the GPU
# =====================================================================
# Why it exists: diag(B) starts at spread ||A||/sqrt(n) and must climb to the
# true spectral spread, which costs ~(1/2)log n sweeps. While the spread is
# small nearly every gap is comparable to every coupling, so thousands of
# pairs saturate at +-pi/4 and the contraction rate sits at 0.99. Diagonalising
# sorted-contiguous blocks exactly hands the iteration ~sqrt(m) of that spread
# for free. On CPU this took 20/24/29 sweeps down to 9/11/14 at m=32.
#
# Correctness: for block-diagonal orthogonal P, the (I,J) block of P^T B P is
# Q_I^T B_IJ Q_J, whose Frobenius norm equals ||B_IJ||_F. Off-block masses are
# preserved individually and within-block off-diagonal mass is zeroed, so a
# block pass is a strict monotone reduction of off(B) for ANY grouping.
#
# GPU shaping: the CPU version loops over blocks in Python, issuing hundreds of
# tiny kernels -- latency-bound. Here the sorted diagonal is rolled cyclically
# by `offset` instead of being cut into a ragged head and tail, so every block
# has size exactly m and ONE batched eigh handles all of them. The grouping
# that changes (extreme-low with extreme-high entries land together) is
# harmless by the paragraph above.

def block_pass(B, X, m, offset):
    """One exact block-Jacobi pass on sorted-contiguous blocks of size m."""
    xp = _am(B)
    n = B.shape[0]
    nb = n // m
    if nb < 2:
        return B, X
    keep = nb * m

    p = xp.argsort(xp.real(xp.diag(B)))
    if offset:
        p = xp.roll(p, -int(offset))
    B = B[p][:, p]
    X = X[:, p]

    sub = B[:keep, :keep].reshape(nb, m, nb, m)
    blocks = sub[xp.arange(nb), :, xp.arange(nb), :]          # (nb, m, m)
    blocks = (blocks + blocks.conj().transpose(0, 2, 1)) / 2.0
    Q = xp.linalg.eigh(blocks)[1]                             # batched

    # Block-diagonal Q as one (keep, keep) operator: applying it is then two
    # ordinary gemms rather than 2*nb small ones.
    Qfull = xp.zeros((keep, keep), dtype=B.dtype)
    Qfull.reshape(nb, m, nb, m)[xp.arange(nb), :, xp.arange(nb), :] = Q

    X[:, :keep] = X[:, :keep] @ Qfull
    B[:, :keep] = B[:, :keep] @ Qfull
    B[:keep, :] = Qfull.conj().T @ B[:keep, :]
    return (B + B.conj().T) / 2.0, X


print("SSJ-BC block pass loaded (batched).")

In [ ]:
# =====================================================================
#  3b - IPT and the hybrid: leave the manifold once the basin opens
# =====================================================================
# IPT is PURE GEMM -- one matmul per iteration, no factorization, no
# retraction -- because it never maintains orthogonality: it pins V_jj = 1
# instead. That only converges inside the basin rho = max|W_ij|/gap_ij < 1,
# which the SSJ sweeps buy. The basin opens as a CLIFF (rho falls through 1
# in a single sweep), so the hybrid watches rho -- O(n^2), free next to a
# gemm -- and hands off the endgame.
#
# Leaving the manifold must be done CLEANLY: one exact QR at the hand-off.
# The coarse globalizing phase may leave the frame orthonormal only to its
# own loose tolerance, and IPT inherits any defect verbatim as a similarity
# error (measured: 1e-8 eigenvalue error without this, 4e-15 with it).

def ipt_rate(B):
    """max |W_ij| / |gap_ij| -- IPT's contraction rate in the frame of B."""
    xp = _am(B)
    n = B.shape[0]
    d = xp.real(xp.diag(B))
    gap = xp.abs(d[None, :] - d[:, None])
    absW = xp.abs(B - xp.diag(xp.diag(B)))
    eye = xp.eye(n, dtype=bool)
    gap = xp.where(eye, 1.0, gap)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = xp.where(eye, 0.0, absW / gap)
    return float(xp.max(ratio))


def ipt_iterate(W, d, max_iter=80, tol=1e-13, norm_A=1.0):
    """IPT fixed point from V = I: Lam_j = d_j + (WV)_jj,
    V_ij = (WV)_ij / (Lam_j - d_i), V_jj = 1. One gemm per iteration;
    the elementwise work is kept in place so it does not rival the gemm."""
    xp = _am(W)
    n = W.shape[0]
    idx = xp.arange(n)
    V = xp.eye(n, dtype=W.dtype)
    R = xp.empty_like(V)
    err0 = None
    for it in range(1, max_iter + 1):
        WV = W.copy() if it == 1 else W @ V
        Lam = d + xp.real(xp.diag(WV))
        xp.subtract(Lam[None, :], d[:, None], out=R)
        R[idx, idx] = 1.0
        xp.reciprocal(R, out=R)
        xp.multiply(WV, R, out=WV)
        WV[idx, idx] = 1.0
        xp.subtract(WV, V, out=V)               # V <- update
        err = float(xp.max(xp.abs(V)))
        V = WV
        if err0 is None:
            err0 = max(err, 1e-300)
        if err < tol * norm_A:
            return V, Lam, it, True
        if err > 1e3 * err0:                    # the basin boundary, seen
            return V, Lam, it, False            # from inside the loop
    return V, Lam, max_iter, False


def ssj_ipt(A, tol=1e-13, gate=0.5, coarse=1e-2, probe_iters=80,
            return_info=False, **ssj_kw):
    """Globalize with SSJ (schedule / blocks / mixed via ssj_kw), hand the
    endgame to IPT. Falls back to finishing with SSJ if the gate never
    opens (exact ties keep rho infinite -- correctly: IPT cannot resolve
    them)."""
    xp = _am(A)
    A = xp.asarray(A)
    n = A.shape[0]
    nrm_f = float(xp.linalg.norm(A, ord="fro"))
    norm_A = nrm_f / max(np.sqrt(n), 1.0)
    X = None
    sweeps = 0
    ipt_iters = 0
    target = coarse
    B = A
    while True:
        w, V, info = ssj_eigh(A, tol=max(tol, target), X0=X,
                              return_info=True, **ssj_kw)
        X = V
        sweeps += info.get("sweeps_total", info["sweeps"])
        B = X.conj().T @ (A @ X)
        B = (B + B.conj().T) / 2.0
        if off_frob(B) <= tol * nrm_f:
            break
        if ipt_rate(B) < gate:
            X = _orth_qr(X)                     # leave the manifold cleanly
            B = X.conj().T @ (A @ X)
            B = (B + B.conj().T) / 2.0
            d = xp.real(xp.diag(B))
            W = B - xp.diag(xp.diag(B))
            Vd, Lam, it, ok = ipt_iterate(W, d, probe_iters, tol, norm_A)
            ipt_iters += it
            if ok:
                Vd = Vd / xp.linalg.norm(Vd, axis=0, keepdims=True)
                Vfull = X @ Vd
                order = xp.argsort(Lam)
                info = {"sweeps": sweeps, "ipt_iters": ipt_iters,
                        "converged": True, "path": "ipt"}
                return ((Lam[order], Vfull[:, order], info)
                        if return_info else (Lam[order], Vfull[:, order]))
            gate *= 0.1                         # IPT failed: raise the bar
        if target <= tol:
            break
        target = max(target * 1e-2, tol)        # SSJ carries on, tighter
    w = xp.real(xp.diag(B))
    order = xp.argsort(w)
    info = {"sweeps": sweeps, "ipt_iters": ipt_iters, "converged": True,
            "path": "ssj"}
    return ((w[order], X[:, order], info) if return_info
            else (w[order], X[:, order]))


print("IPT + hybrid loaded.")

In [ ]:
# =====================================================================
#  4 - Timing harness and problem generators
# =====================================================================
import time


def sync(xp):
    if xp is not np:
        xp.cuda.Device().synchronize()


def timed(fn, xp, reps=3):
    """Min-of-reps wall time. The synchronize() calls are not optional: without
    them you measure kernel LAUNCH time, which for a fast GPU op is ~10 us
    regardless of the work queued behind it."""
    fn()                                    # warm up: autotune, workspaces, JIT
    sync(xp)
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        sync(xp)
        best = min(best, time.perf_counter() - t0)
    return best


def goe(n, xp, seed=0, dtype=None):
    """GOE: flat spectrum, the hardest case for SSJ (no graded structure to
    exploit, so the prologue lever does nothing)."""
    rng = xp.random.default_rng(seed)
    M = rng.standard_normal((n, n))
    A = (M + M.T) / xp.sqrt(xp.asarray(2.0 * n))
    return A.astype(dtype or np.float64)


def check(A, w, V, xp):
    """Residual and orthogonality, both relative to ||A||_F.

    A fast wrong answer is still wrong, so every timed configuration below
    reports these alongside its wall time.
    """
    resid = float(xp.linalg.norm(A @ V - V * w, ord="fro")) / float(
        xp.linalg.norm(A, ord="fro"))
    ortho = float(xp.linalg.norm(
        V.conj().T @ V - xp.eye(A.shape[0], dtype=V.dtype), ord="fro"))
    return resid, ortho


print("harness loaded.")

In [ ]:
# =====================================================================
#  5 - The GPU sweep: SSJ configurations vs cuSOLVER syevd
# =====================================================================
SIZES = [512, 1024, 2048]        # add 4096 if the card has memory and patience

# Labels are shared with the CPU control in cell 6 so the two tables can be
# joined on (n, label) without any string surgery. Configs depend on n (the
# schedule scales with it), so they are built per size.
BASE = "gemm f64"
BASE_SCHED = "gemm f64 +sched"
HYB_SCHED = "hybrid f64 +sched"


def make_configs(n):
    """(label, solver, kwargs) rows. solver: 'ssj' -> ssj_eigh,
    'hyb' -> ssj_ipt. The schedule [n/2, n/4, 32] injects diagonal spread
    up front (15 -> 8 sweeps at n=800 on CPU); the hybrid hands the endgame
    to pure-gemm IPT at the basin cliff; mixed runs the globalization in
    fp32, which is where tensor-core cards should shine."""
    sched = [n // 2, n // 4, 32]
    bc = dict(block_fn=block_pass)
    return [
        (BASE,              "ssj", dict(method="gemm")),
        ("gemm f64 +BC32",  "ssj", dict(method="gemm", block_m=32, **bc)),
        (BASE_SCHED,        "ssj", dict(method="gemm", block_m=sched, **bc)),
        ("gemm mixed +sched", "ssj", dict(method="gemm", precision="mixed",
                                          block_m=sched, **bc)),
        (HYB_SCHED,         "hyb", dict(method="gemm", block_m=sched, **bc)),
        ("hybrid mixed +sched", "hyb", dict(method="gemm", precision="mixed",
                                            block_m=sched, **bc)),
    ]


gpu_ratio, gpu_rows = {}, []
for n in SIZES:
    A = goe(n, cp, seed=1)
    t_gemm = timed(lambda: A @ A, cp, reps=5)
    t_ref = timed(lambda: cp.linalg.eigh(A), cp, reps=3)
    print(f"\n=== n={n} ===")
    print(f"  fp64 gemm       {t_gemm*1e3:9.2f} ms")
    print(f"  cuSOLVER eigh   {t_ref*1e3:9.2f} ms  "
          f"= {t_ref/t_gemm:6.1f} gemm-equivalents"
          f"   (plain SSJ needs ~55-80; the composed solver ~25-40)")
    print(f"  {'config':>21}{'ms':>10}{'vs cuSOLVER':>13}"
          f"{'sweeps':>8}{'ipt it':>7}{'resid':>10}{'ortho':>10}")

    for label, kind, kw in make_configs(n):
        solver = ssj_eigh if kind == "ssj" else ssj_ipt
        try:
            t = timed(lambda: solver(A, **kw), cp, reps=2)
            w, V, info = solver(A, return_info=True, **kw)
            sync(cp)
            resid, ortho = check(A, w, V, cp)
            sw = info.get("sweeps_total", info["sweeps"])
            it = info.get("ipt_iters", 0)
            flag = "" if info["converged"] else "  NOT CONVERGED"
            print(f"  {label:>21}{t*1e3:10.1f}{t/t_ref:12.1f}x{sw:8d}{it:7d}"
                  f"{resid:10.1e}{ortho:10.1e}{flag}")
            gpu_ratio[(n, label)] = t / t_ref
            gpu_rows.append((n, label, t, t_ref, t / t_ref, sw, resid, ortho))
        except Exception as e:
            print(f"  {label:>21}  FAILED: {type(e).__name__}: {e}")

print("\nDone. `gpu_rows` / `gpu_ratio` hold the raw numbers.")

In [ ]:
# =====================================================================
#  6 - CPU control: the same code, so the answer is a SHIFT in the ratio
# =====================================================================
# Absolute times on a Colab vCPU are meaningless. The ratio SSJ/LAPACK is not:
# it is the same quantity measured on the GPU against cuSOLVER, so the
# difference between the two ratios is the actual result of this notebook.

CPU_SIZES = [512, 1024]          # the vCPU is slow; keep this modest

gpu_ratio = globals().get("gpu_ratio", {})   # so this cell stands alone too
cpu_ratio = {}
print(f"  {'n':>6}{'config':>21}{'SSJ ms':>10}{'LAPACK ms':>12}{'ratio':>9}"
      f"{'sweeps':>8}{'resid':>10}")
for n in CPU_SIZES:
    A = goe(n, np, seed=1)
    t_ref = timed(lambda: np.linalg.eigh(A), np, reps=3)
    for label, kind, kw in make_configs(n):
        if label not in (BASE, BASE_SCHED, HYB_SCHED):
            continue                     # CPU control: the three key rows
        solver = ssj_eigh if kind == "ssj" else ssj_ipt
        t = timed(lambda: solver(A, **kw), np, reps=2)
        w, V, info = solver(A, return_info=True, **kw)
        resid, _ = check(A, w, V, np)
        cpu_ratio[(n, label)] = t / t_ref
        sw = info.get("sweeps_total", info["sweeps"])
        print(f"  {n:6d}{label:>21}{t*1e3:10.1f}{t_ref*1e3:12.1f}"
              f"{t/t_ref:8.1f}x{sw:8d}{resid:10.1e}")

# ------------------------------------------------------------------ verdict
print("\n" + "=" * 68)
print("  THE NUMBER THIS NOTEBOOK EXISTS TO PRODUCE")
print("=" * 68)
print(f"  {'n':>6}{'config':>18}{'CPU ratio':>12}{'GPU ratio':>12}{'shift':>10}")
shown = 0
for (n, label), rc in sorted(cpu_ratio.items()):
    rg = gpu_ratio.get((n, label))
    if rg is None:
        continue
    shown += 1
    print(f"  {n:6d}{label:>18}{rc:11.1f}x{rg:11.1f}x{rc/rg:9.2f}x")
if not shown:
    print("  (no matching GPU rows -- run cell 5 first)")

print("""
  shift = (SSJ/LAPACK on CPU) / (SSJ/cuSOLVER on GPU)

    > 1   the GPU narrows SSJ's disadvantage by that factor
    ~ 1   SSJ's all-gemm structure buys nothing here; the loss is flop count
    < 1   cuSOLVER exploits this GPU better than SSJ does

  A shift near 1 measured on an fp64-crippled card (see cell 1) is NOT
  conclusive -- rerun on an A100 or V100 before drawing any conclusion. A
  shift near 1 on a card with real fp64 throughput IS conclusive, and is a
  genuine negative result worth recording as one.""")

In [ ]:
# =====================================================================
#  7 - Warm-started tracking: the regime where SSJ can actually WIN
# =====================================================================
# A cold full solve is the HARDEST case for SSJ. The favourable case is
# re-solving a matrix that moved slightly -- molecular dynamics, parameter
# sweeps, continuation, self-consistent field loops. SSJ accepts the previous
# eigenbasis as X0 and lands directly in the quadratic tail (1-5 sweeps
# instead of 20-29). syevd has no way to accept a warm start: it pays full
# price every single time.
#
# A ratio BELOW 1.0 in this table is SSJ beating cuSOLVER outright.

TRACK_SIZES = [1024, 2048]
EPSILONS = [1e-2, 1e-4, 1e-6]

print(f"  {'n':>6}{'eps':>8}{'SSJ warm ms':>13}{'cuSOLVER ms':>13}"
      f"{'ratio':>9}{'sweeps':>8}{'resid':>10}")
for n in TRACK_SIZES:
    A = goe(n, cp, seed=1)
    _, V0 = cp.linalg.eigh(A)               # the "previous" eigenbasis
    rng = cp.random.default_rng(5)
    P = rng.standard_normal((n, n))
    P = (P + P.T) / 2.0
    P = P / float(cp.linalg.norm(P, ord="fro"))

    for eps in EPSILONS:
        A2 = A + eps * P
        t_ref = timed(lambda: cp.linalg.eigh(A2), cp, reps=2)
        t_warm = timed(lambda: ssj_eigh(A2, X0=V0, method="gemm"), cp, reps=2)
        w, V, info = ssj_eigh(A2, X0=V0, method="gemm", return_info=True)
        sync(cp)
        resid, _ = check(A2, w, V, cp)
        mark = "   <-- SSJ WINS" if t_warm < t_ref else ""
        print(f"  {n:6d}{eps:8.0e}{t_warm*1e3:13.1f}{t_ref*1e3:13.1f}"
              f"{t_warm/t_ref:8.2f}x{info['sweeps']:8d}{resid:10.1e}{mark}")

print("""
If the ratio is below 1 here but above 1 in cell 5, the honest summary is:
"SSJ is not a general-purpose replacement for syevd, but it is the faster
option for tracking a slowly-varying matrix." That is a narrower claim than
the campaign started with, and a defensible one.""")

In [ ]:
# =====================================================================
#  8 - The OTHER pure-gemm family: purification (SP2) + the refinement ladder
# =====================================================================
# This is the family the campaign converged on after the SSJ cells above were
# written, and it is the more GPU-shaped of the two: every flop outside the
# leaves is a full-rate gemm, and SP2 is ONE gemm per iteration, globally
# convergent, ~30 iterations regardless of n.
#
# On CPU it LOSES to dsyevd (0.13-0.17x) -- but that is a verdict about a
# machine where gemm and syevd are both well tuned. Two CPU-specific findings
# are what this cell exists to retest:
#
#   * "reduced precision pays only through gemm": on that box sgemm is 2x
#     dgemm, but ssyevd, sgeqrf and ssytrf all run at fp64 SPEED. A GPU with
#     a 14:1 fp32:fp64 ratio breaks that assumption completely.
#   * the refinement ladder (coarse supplier + ~5 gemms per squared digit)
#     did not pay on CPU for exactly that reason -- the cheap coarse solve
#     did not exist. On a GPU it does.
#
# This port is deliberately faithful: it reproduces the CPU implementation's
# numbers exactly on the NumPy path (validated against the repo across GOE,
# exact 5-fold ties, 1e-9 clusters and zero-diagonal, both precisions). The
# one structural change is that the random range-finder matrix is cached per
# (n, backend). A tempting sync-removal in SP2 was tried and rejected -- see
# sp2_projector's docstring.

_G_CACHE = {}


def _bounds_gersh(A):
    """Gershgorin: an EXACT enclosure, O(n^2). Loose (12.5x on GOE n=800) but
    never wrong, which is what makes it the fallback."""
    xp = _am(A)
    d = xp.real(xp.diag(A))
    r = xp.sum(xp.abs(A), axis=1) - xp.abs(d)
    return float(xp.min(d - r)), float(xp.max(d + r))


def _bounds_tight(A, iters=7):
    """Power iteration on A^2 (robust to the +-lambda near-ties of a flat
    spectrum), inflated 25%, clipped to Gershgorin.

    The SP2 seed slope is inversely proportional to the enclosure width, so a
    12.5x-loose enclosure costs ~4 extra doubling iterations per split. This
    is an ESTIMATE, not an enclosure -- an under-enclosure spills eigenvalues
    outside [0,1] where SP2's squaring branch diverges, which is why the
    caller carries a divergence guard and a Gershgorin retry.
    """
    xp = _am(A)
    v = 1.0 + 0.01 * xp.arange(A.shape[0], dtype=_rdtype(A))
    v = v / xp.linalg.norm(v)
    est = 0.0
    for _ in range(iters):
        w = A @ (A @ v)
        est = float(xp.linalg.norm(w))
        if est == 0.0:
            return _bounds_gersh(A)
        v = w / est
    m = 1.25 * float(np.sqrt(est))
    glo, ghi = _bounds_gersh(A)
    return max(-m, glo), min(m, ghi)


def sp2_projector(A, mu, tol=1e-12, max_iter=100, warmup=6, dtype=None):
    """Spectral projector below mu: McWeeny warmup, then SP2 (Niklasson).

    SP2 is P <- P^2 or 2P - P^2, branched on the trace against the target
    rank -- ONE gemm per iteration against McWeeny's two.

    KNOWN GPU COST, deliberately not "fixed" here: the trace branch reads a
    host float every iteration, ~30 syncs per projector. The obvious removal
    is to blend the branch on-device,

        P <- P2 + 2s(P - P2),   s = 1 for 2P - P^2, else 0,

    which is algebraically identical. It was tried and REJECTED: `2.0 * s` is
    a float64 scalar, so float64 * float32_array silently promotes the whole
    fp32 projector to fp64 -- "mixed" stops being mixed, and the split then
    desynchronises from the `net` and polish tolerances calibrated for an
    fp32-quality split. Measured on exact 5-fold ties at n=400: the blend
    produced a TIGHTER projector (1.5e-08 vs 5.3e-06) and a WORSE answer
    (dlam 1.3e-12 vs 3.7e-15, orthogonality 1.0e-06 vs 1.5e-11).

    A dtype-correct blend is probably fine, but it would need its own
    validation on a card, and this campaign does not ship unmeasured kernels
    (OPTIMIZATION_LOG #7). The syncs stay until someone measures the alternative.
    """
    xp = _am(A)
    n = A.shape[0]
    r = 0
    P = None
    for bounds_fn in (_bounds_tight, _bounds_gersh):
        lo, hi = bounds_fn(A)
        c = 0.5 / max(hi - mu, mu - lo, 1e-300)
        P = -c * A + (0.5 + c * mu) * xp.eye(n, dtype=A.dtype)
        if dtype is not None:
            P = P.astype(dtype)
        for _ in range(warmup):                   # McWeeny
            P2 = P @ P
            P = 3.0 * P2 - 2.0 * (P @ P2)
        tr = float(xp.trace(P.astype(np.float64)))
        if not np.isfinite(tr):
            continue                              # under-enclosure: retry
        r = float(np.rint(tr))
        ok = True
        prev_err = np.inf
        for it in range(max_iter):
            P2 = P @ P
            if it % 3 == 0:
                # O(n^2) check, and it doubles as the divergence guard: an
                # under-enclosed eigenvalue survives the McWeeny warmup (whose
                # attractor basin reaches ~1.37) and explodes only later under
                # SP2's squaring branch.
                err = float(xp.linalg.norm(P2 - P, ord="fro"))
                if err < tol * np.sqrt(n):
                    break
                if not np.isfinite(err) or err > 1e3 * prev_err:
                    ok = False
                    break
                prev_err = err
            if float(xp.trace(P)) - r > 0:
                P, P2 = P2, P                     # P <- P^2 is a swap
            else:
                P = 2.0 * P - P2
        if ok:
            return P.astype(np.float64), int(r)
    return P.astype(np.float64), int(r)


def ipt_polish(A, w, V):
    """One consult-A IPT step in the near-eigenbasis: 3 gemms + elementwise.

    Purification's structural flaw is that its map never consults A after the
    seed, so ANY projector is a fixed point and split-boundary mixing survives
    to the answer. IPT is nothing but consulting A, and in the purified basis
    rho ~ 0, so one step lands the residual at ~3e-15. Near-degenerate pairs
    are guarded: |gap| < 1e3|W| is exactly |C| > 1e-3, and phrasing the mask
    on C absorbs the inf (gap~0) and NaN (0/0) cases together.
    """
    xp = _am(A)
    n = A.shape[0]
    B = V.T @ (A @ V)
    B = (B + B.T) * 0.5
    d = xp.diag(B).copy()
    gap = d[None, :] - d[:, None]
    gap = gap + xp.eye(n, dtype=gap.dtype)        # neutralize the diagonal
    W = B - xp.diag(d)
    with np.errstate(divide="ignore", invalid="ignore"):
        C = W / gap
    C = xp.where(xp.abs(C) <= 1e-3, C, 0.0)
    C = C - xp.diag(xp.diag(C)) + xp.eye(n, dtype=C.dtype)
    V2 = V @ C
    V2 = V2 / xp.linalg.norm(V2, axis=0, keepdims=True)
    order = xp.argsort(d)
    return d[order], V2[:, order]


def refine_eigh(A, w, V, pairs=2):
    """Upgrade ANY approximate eigenbasis of Hermitian A to fp64 accuracy.

    The refinement ladder: alternate one consult-A IPT step (3 gemms, fixes
    the residual, first-order and NON-orthogonal) with one Newton-Schulz step
    (2 gemms, clears the O(err^2) orthogonality defect the polish leaves,
    which would otherwise floor the next step). Skip either half and it
    floors. Each pair roughly squares the error.

    MEASURED BASIN: converges from coarse error up to ~1e-3..1e-4 and stalls
    proportionally beyond. It buys the last 7-11 digits, never the first four
    -- those must come from a real solver. On a GPU that solver is an fp32
    cuSOLVER eigh, which is the whole point of cell 9.
    """
    xp = _am(A)
    n = A.shape[0]
    V = V.astype(A.dtype, copy=True)
    eye = xp.eye(n, dtype=A.dtype)
    for _ in range(pairs):
        w, V = ipt_polish(A, w, V)
        G = V.T @ V
        V = V @ (1.5 * eye - 0.5 * G)
    return w, V


def purify_eigh(A, leaf=None, tol=1e-12, polish=True, precision="full",
                _depth=0):
    """Full symmetric spectrum by recursive purification bisection.

    Each level builds the projector below mu = trace/n by SP2, extracts an
    orthonormal split basis with a randomized range-finder (P is idempotent
    to tol, so QR([P G1, (I-P) G2]) splits exactly -- one unpivoted QR, no
    dgeqp3), and recurses. Leaves fall to the vendor eigensolver.

    Three guards, all free on the happy path and all earned by a measured
    failure: a divergence guard with Gershgorin retry inside SP2; an
    off-block-mass net after the split (eigenvalues exactly AT mu sit at the
    purification fixed point 1/2, where the trace branch can mis-rank); and a
    boundary audit after recursion (a split landing inside a ~1e-9 cluster
    leaves each block holding a fragment, and no polish can reunite them).
    """
    xp = _am(A)
    n = A.shape[0]
    if leaf is None:
        leaf = max(64, n // 2)
    if n <= leaf:
        return xp.linalg.eigh(A)
    mixed = precision == "mixed"
    low = np.float32 if mixed else None
    mu = float(xp.trace(A)) / n
    P, r = sp2_projector(A, mu, tol=(1e-6 if mixed else tol), dtype=low)
    if r <= 0 or r >= n:
        return xp.linalg.eigh(A)

    key = (n, xp.__name__)
    if key not in _G_CACHE:
        _G_CACHE[key] = xp.asarray(
            np.random.default_rng(0x5D1).standard_normal((n, n)))
    G = _G_CACHE[key].astype(A.dtype, copy=False)

    Y = xp.empty((n, n), dtype=A.dtype)
    Y[:, :r] = P @ G[:, :r]
    Y[:, r:] = G[:, r:] - P @ G[:, r:]
    Q = xp.linalg.qr(Y)[0]
    B = Q.T @ (A @ Q)
    B = (B + B.T) * 0.5

    net = 1e-4 if mixed else 1e-8      # fp32 splits legitimately carry ~1e-7
    if float(xp.linalg.norm(B[r:, :r], ord="fro")) > \
            net * float(xp.linalg.norm(A, ord="fro")):
        return xp.linalg.eigh(A)

    w1, V1 = purify_eigh(B[:r, :r], leaf, tol, False, precision, _depth + 1)
    w2, V2 = purify_eigh(B[r:, r:], leaf, tol, False, precision, _depth + 1)
    V = xp.empty((n, n), dtype=A.dtype)
    V[:, :r] = Q[:, :r] @ V1
    V[:, r:] = Q[:, r:] @ V2

    if w1.size and w2.size:
        b1, b2 = float(w1.max()), float(w2.min())
        if abs(b2 - b1) < 1e-7 * max(abs(b1), abs(b2), 1e-300):
            return xp.linalg.eigh(A)   # split cut inside a tight cluster

    w = xp.concatenate([w1, w2])
    order = xp.argsort(w)
    w, V = w[order], V[:, order]
    if polish:
        # fp64 splits need one consult-A step; fp32 splits need two, with a
        # Newton-Schulz re-orthonormalization BETWEEN them (never after: the
        # final polish leaves a defect of (its input error)^2 ~ 1e-16, so a
        # trailing NS step is 2 wasted gemms).
        pairs = 2 if mixed else 1
        eye = xp.eye(n, dtype=A.dtype)
        for k in range(pairs):
            w, V = ipt_polish(A, w, V)
            if mixed and k < pairs - 1:
                G2 = V.T @ V
                V = V @ (1.5 * eye - 0.5 * G2)
    return w, V


print("purification (SP2) + refinement ladder loaded.")


In [ ]:
# =====================================================================
#  9 - COARSE + LADDER: the design the campaign actually converged on
# =====================================================================
# Everything above races a whole solver against cuSOLVER. This cell races an
# ARCHITECTURE, and it is the one measurement this notebook exists for that
# the CPU genuinely cannot make.
#
# The canonical structure: every solver here is a COARSE SUPPLIER plus the
# REFINEMENT LADDER. The ladder squares the error per ~5-gemm pair but its
# basin is small (coarse error <~ 1e-3..1e-4), so it buys the last 7-11
# digits and never the first four. The first four must come from a real
# solver -- and the cheaper that solver, the better the whole thing looks.
#
# On CPU this design DOES NOT PAY, for one reason: ssyevd runs at dsyevd
# SPEED there (measured 1.01x/0.96x), so the cheap coarse supplier does not
# exist and the ladder is pure overhead (0.65-0.99x against simply calling
# dsyevd). That is the single most substrate-specific finding of the whole
# campaign, and cell 1 has already told you this card breaks its premise:
# fp32 gemm runs many times fp64 here.
#
# So the question is exactly: does an fp32 cuSOLVER solve plus an fp64 ladder
# beat an fp64 cuSOLVER solve? A ratio above 1 is the campaign's design
# winning on the substrate it was predicted to win on.
#
# Accuracy is asserted on every row before its time is believed: the ladder's
# whole claim is fp64 accuracy from an fp32 start, so a row that is fast and
# inaccurate is not a result, it is a bug.

PUR_SIZES = [512, 1024, 2048]


def coarse_ladder(A, pairs=2):
    """fp32 cuSOLVER coarse solve, then the fp64 ladder. ~5 gemms per pair."""
    xp = _am(A)
    w32, V32 = xp.linalg.eigh(A.astype(np.float32))
    return refine_eigh(A, w32.astype(A.dtype), V32.astype(A.dtype), pairs=pairs)


print(f"  {'n':>6}{'method':>22}{'ms':>10}{'vs fp64 eigh':>14}"
      f"{'gemm-eq':>9}{'resid':>10}{'ortho':>10}")
pur_rows = []
for n in PUR_SIZES:
    A = goe(n, cp, seed=1)
    t_gemm = timed(lambda: A @ A, cp, reps=5)
    t_ref = timed(lambda: cp.linalg.eigh(A), cp, reps=3)

    # the incumbent's own cost in the unit that decides everything
    print(f"\n  n={n}: fp64 gemm {t_gemm*1e3:.2f} ms | cuSOLVER fp64 eigh "
          f"{t_ref*1e3:.2f} ms = {t_ref/t_gemm:.1f} gemm-equivalents")
    t32 = timed(lambda: cp.linalg.eigh(A.astype(np.float32)), cp, reps=3)
    print(f"        cuSOLVER fp32 eigh {t32*1e3:.2f} ms "
          f"= {t_ref/t32:.2f}x cheaper than fp64"
          f"   <-- on the CPU this ratio was 1.0, which is why the ladder lost")

    cands = [
        ("cuSOLVER fp64 (ref)", lambda: cp.linalg.eigh(A)),
        ("purify full",         lambda: purify_eigh(A)),
        ("purify mixed",        lambda: purify_eigh(A, precision="mixed")),
        ("fp32 eigh + ladder",  lambda: coarse_ladder(A, pairs=2)),
        ("fp32 eigh + ladder x3", lambda: coarse_ladder(A, pairs=3)),
    ]
    for label, fn in cands:
        try:
            w, V = fn()
            sync(cp)
            resid, ortho = check(A, w, V, cp)
            if not (resid < 1e-11 and ortho < 1e-9):
                print(f"  {n:6d}{label:>22}{'':>10}{'':>14}{'':>9}"
                      f"{resid:10.1e}{ortho:10.1e}  NOT ACCURATE - not timed")
                continue
            t = timed(fn, cp, reps=2)
            # the reference row is cuSOLVER timed against its own stored
            # time, so it "wins" by pure noise -- never mark it
            mark = ("   <-- BEATS cuSOLVER"
                    if (t < t_ref and label != "cuSOLVER fp64 (ref)") else "")
            print(f"  {n:6d}{label:>22}{t*1e3:10.1f}{t_ref/t:13.2f}x"
                  f"{t/t_gemm:9.1f}{resid:10.1e}{ortho:10.1e}{mark}")
            pur_rows.append((n, label, t, t_ref, t_ref / t, resid, ortho))
        except Exception as e:
            print(f"  {n:6d}{label:>22}  FAILED: {type(e).__name__}: {e}")

print("""
  How to read this table

  The 'fp32 eigh + ladder' rows are the campaign's design. Above 1.00x they
  beat cuSOLVER's own fp64 solve, using cuSOLVER itself as the coarse
  supplier -- which would be the first substrate on which coarse+ladder pays,
  and a direct consequence of the fp32:fp64 ratio cell 1 measured.

  If they land BELOW 1.00x, read the 'cuSOLVER fp32 eigh' line above them
  before concluding anything: if fp32 eigh is not meaningfully cheaper than
  fp64 eigh on this card, the premise failed rather than the design, and the
  result is the GPU restating the CPU's substrate ambush #6 rather than a new
  finding.

  The 'purify' rows are the whole-solver comparison. They lose 6-8x on CPU;
  what matters here is the DIRECTION of the change, and the gemm-equivalents
  column is how to see it without wall times entering into it at all.

  READ THE gemm-eq COLUMN AGAINST THE CARD, NOT IN ISOLATION. It is
  t_method / t_fp64_gemm, so an fp64-CRIPPLED card inflates the denominator
  and DEFLATES every gemm-equivalent count on the page -- including
  cuSOLVER's. Cell 1 told you the fp32:fp64 ratio; if it is far above 2, then
  a small cuSOLVER gemm-equivalent count here does not mean cuSOLVER is
  gemm-cheap in general, it means fp64 gemm is artificially slow on this die.
  On a card with real fp64 throughput the same solve costs the same seconds
  but many MORE gemm-equivalents, which is the regime where a fixed
  ~5-gemms-per-pair ladder can fit underneath it. A ladder verdict measured
  here is a verdict about this card.""")


In [ ]:
# =====================================================================
#  10 - SDC: spectral divide and conquer for NON-symmetric A
# =====================================================================
# Everything above is symmetric, where cuSOLVER's syevd turned out to be a
# very strong incumbent (5.4 gemm-equivalents at n=2048, falling toward its
# flop floor). The nonsymmetric problem is a different contest for a reason
# that has nothing to do with the algorithm:
#
#   * on CPU, dgeev costs 131-181 gemm-equivalents against dsyevd's 17-25 --
#     the incumbent is ~7x weaker in exactly the unit that decided the
#     symmetric race;
#   * and cuSOLVER appears to provide NO general nonsymmetric eigensolver at
#     all (syevd/syevj/sygvd and the SVDs, but no geev). Cell 12 checks this
#     rather than assuming it. If it holds, the GPU comparison is not "beat a
#     tuned vendor routine" but "beat a round trip to the host".
#
# SDC has op-count PARITY with dgeev (88 gemm-equivalents against 89, measured
# at n=400), and every transformation it applies is an orthogonal similarity,
# so it has no basin condition at all -- unlike IPT, SSJ and the shears, each
# of which stalls or diverges on non-normal input.
#
# TWO PORT CHANGES, both forced and both interesting.
#
# 1. THE SPLIT BASIS. The CPU version uses pivoted QR (dgeqp3) to extract an
#    orthonormal basis for range(P). CuPy's qr has no pivoting. The fix is the
#    randomized range-finder already validated in the purification family:
#    QR([P G1, (I-P) G2]) with ONE unpivoted QR. Column ORDER is load-bearing
#    here -- OPTIMIZATION_LOG #24 records building the basis from a pivoted QR of
#    [P, I-P] instead, whose column reordering destroys the range separation
#    and reported a bogus ||A21|| = 2.6e-01 on a symmetric matrix.
#
# 2. THE LEAF. The CPU version recurses to ~3n/5 and hands both halves to
#    dgeev. With no geev on the device that option does not exist, so this
#    port offers both and lets the cell MEASURE which wins:
#      leaf_solver="host"  -- copy each leaf to the CPU, numpy.linalg.eig,
#                             copy back. One transfer per leaf.
#      leaf_solver="deep"  -- recurse all the way to 2x2 and solve in closed
#                             form, entirely on the device.
#    OPTIMIZATION_LOG #25 measured "deep" as 4x SLOWER on CPU -- but that verdict
#    assumed a cheap leaf solver existed, which is exactly the assumption a
#    GPU breaks. Deep recursion is also cheaper than it looks: level k holds
#    2^k blocks of size n/2^k, so the sign work sums to ~4/3 of the top-level
#    split. What deep recursion really costs is kernel LAUNCHES on small
#    blocks, which is the #17 leaf lesson in its GPU form.
#
# Complex pairs are safe under this splitter: a conjugate pair shares a real
# part, so a vertical cut at Re(z) = sigma never separates one.


if "_G_CACHE" not in globals():   # cell 9 defines it; stand alone too
    _G_CACHE = {}


def _sign_iterate(X, ns_frob=1.0, tol=1e-12, max_iter=60):
    """Matrix sign by scaled Newton with a Newton-Schulz endgame.

        scaled Newton   X <- (mu X + mu^-1 X^-1)/2,  mu = |det X|^(-1/n)
        Newton-Schulz   X <- X(3I - X^2)/2                    (2 gemms)

    Returns (S, iters, ok). Two hard-won details from the CPU campaign:

    HANDOFF IN THE RIGHT NORM (OPTIMIZATION_LOG #31). NS converges only inside
    ||I - X^2||_2 < 1. Testing a FIXED threshold on ||I - X^2||_F/sqrt(n) --
    an RMS quantity that sits far below the operator norm -- made a SYMMETRIC
    matrix enter NS outside its region and never converge: 2712 ms against
    dgeev's 48 ms at n=400, 13448 ms at n=800. Since ||M||_2 <= ||M||_F,
    gating on ||I - X^2||_F < 1 is guaranteed safe, and in normalized units
    that is a 1/sqrt(n) SCALING LAW, not a constant.

    THE FAR-FIELD GEMM IS OPTIONAL (OPTIMIZATION_LOG #30). A Newton step is X^2 (2n^3)
    plus the inverse (2n^3), and half of it is a gemm whose only job is to
    evaluate the convergence test. But Delta = (X^-1 - X)/2 = X^-1(I - X^2)/2,
    so the update norm is the same signal for O(n^2). Measured: Delta tracks
    dev/2 to two digits through the endgame and is never small while dev is
    large, so gating on it reproduces the same handoff while forming X^2 once
    or twice instead of 8-11 times.
    """
    xp = _am(X)
    n = X.shape[0]
    sqn = np.sqrt(n)
    thresh = ns_frob / sqn
    eye = xp.eye(n, dtype=X.dtype)

    nrm = float(xp.linalg.norm(X, ord="fro")) / sqn
    if nrm == 0.0:
        return X, 0, False
    X = X / nrm

    delta_prev = np.inf
    since_check = 0
    for it in range(1, max_iter + 1):
        if delta_prev < thresh or since_check >= 8:
            since_check = 0
            X2 = X @ X
            dev = float(xp.linalg.norm(X2 - eye, ord="fro")) / sqn
            if not np.isfinite(dev):
                return X, it, False
            if dev < tol:
                return X, it, True
            if dev < thresh:
                X = X @ (1.5 * eye - 0.5 * X2)
                delta_prev = 0.0
                continue
        else:
            since_check += 1
        # scaled Newton; one slogdet + one inv (cuSOLVER getrf/getri)
        sgn, logabsdet = xp.linalg.slogdet(X)
        logabsdet = float(logabsdet)
        if not np.isfinite(logabsdet):
            return X, it, False                    # singular iterate
        Xi = xp.linalg.inv(X)
        mu = np.exp(-logabsdet / n)
        Xn = 0.5 * (mu * X + Xi / mu)
        delta_prev = float(xp.linalg.norm(Xn - X, ord="fro")) / sqn
        X = Xn
        if not np.isfinite(delta_prev):
            return X, it, False
    return X, max_iter, False


def _split_once(A, shift, ns_frob=1.0, tol=1e-12, seed=0x5D1):
    """One spectral split at Re(z) = shift.

    Returns (B, r) with B = Q^T A Q block upper triangular, or (None, code):
    -1 degenerate rank, -2 sign failed, -3 backward error too large.
    """
    xp = _am(A)
    n = A.shape[0]
    S, its, ok = _sign_iterate(A - shift * xp.eye(n, dtype=A.dtype),
                               ns_frob=ns_frob, tol=tol)
    if not ok:
        return None, -2
    P = 0.5 * (xp.eye(n, dtype=A.dtype) + S)
    r = int(np.rint(float(xp.trace(P))))
    if r <= 0 or r >= n:
        return None, -1

    # randomized range-finder, replacing pivoted QR: the FIRST r columns must
    # come from P and the rest from I - P, in that order (OPTIMIZATION_LOG #24)
    key = (n, xp.__name__, "sdc")
    if key not in _G_CACHE:
        _G_CACHE[key] = xp.asarray(
            np.random.default_rng(seed).standard_normal((n, n)))
    G = _G_CACHE[key].astype(A.dtype, copy=False)
    Y = xp.empty((n, n), dtype=A.dtype)
    Y[:, :r] = P @ G[:, :r]
    Y[:, r:] = G[:, r:] - P @ G[:, r:]
    Q = xp.linalg.qr(Y)[0]
    B = Q.T @ (A @ Q)

    # the (2,1) block is zero in exact arithmetic; how far it misses IS the
    # split's backward error, and for nonsymmetric A it scales with the
    # OBLIQUE projector norm ||P||, which is 1 only in the symmetric case
    # (OPTIMIZATION_LOG #24: ||P|| 3.2 -> 1.5e5 as cond(X) runs 10 -> 1e6)
    off = float(xp.linalg.norm(B[r:, :r], ord="fro"))
    if off > 1e-6 * float(xp.linalg.norm(A, ord="fro")):
        return None, -3
    return B, r


def _leaf_2x2(M):
    """Closed-form eigenvalues of a 1x1 or 2x2 block, on the device."""
    xp = _am(M)
    n = M.shape[0]
    if n == 1:
        return xp.asarray(M[0, 0], dtype=np.complex128).reshape(1)
    a, b = M[0, 0], M[0, 1]
    c, d = M[1, 0], M[1, 1]
    tr, det = a + d, a * d - b * c
    disc = xp.asarray(tr * tr / 4.0 - det, dtype=np.complex128)
    root = xp.sqrt(disc)
    return xp.stack([tr / 2.0 + root, tr / 2.0 - root])


def sdc_eigvals(A, min_block=None, leaf_solver="host", ns_frob=1.0,
                tol=1e-12, _depth=0, _stats=None):
    """Eigenvalues of a general real matrix by spectral divide and conquer.

    min_block   : recurse until blocks are this small. Default 3n/5 for
        leaf_solver="host" and 2 for "deep". 3n/5 is measured, not chosen:
        n/2 is too SMALL because the centred split returns r near but never on
        n/2, so one half comes back a few rows too big and buys a whole second
        full-size sign iteration (OPTIMIZATION_LOG #28, worth 1.10x-1.16x).
    leaf_solver : "host" copies leaves to the CPU for numpy.linalg.eig;
        "deep" recurses to 2x2 and never leaves the device.
    """
    xp = _am(A)
    n = A.shape[0]
    if _stats is None:
        _stats = {"splits": 0, "sign_calls": 0, "fallbacks": 0, "leaves": 0}
    if min_block is None:
        min_block = 2 if leaf_solver == "deep" else max(2, 3 * n // 5)

    if n <= min_block or _depth >= 64:
        _stats["leaves"] += 1
        if leaf_solver == "deep" or n <= 2:
            if n <= 2:
                return _leaf_2x2(A), _stats
            # deep mode but block still >2 with no split available
            return xp.asarray(np.linalg.eigvals(_to_host(A)),
                              dtype=np.complex128), _stats
        return xp.asarray(np.linalg.eigvals(_to_host(A)),
                          dtype=np.complex128), _stats

    centre = float(xp.trace(A)) / n
    spread = float(xp.linalg.norm(A, ord="fro")) / np.sqrt(n)
    rng = np.random.default_rng(0xC0FFEE + _depth)

    B = None
    r = -1
    for attempt in range(12):
        shift = centre if attempt == 0 else \
            centre + spread * float(rng.standard_normal()) * 0.5 ** (attempt // 4)
        B, r = _split_once(A, shift, ns_frob=ns_frob, tol=tol)
        _stats["sign_calls"] += 1
        if B is not None:
            break
    if B is None:
        _stats["fallbacks"] += 1
        return xp.asarray(np.linalg.eigvals(_to_host(A)),
                          dtype=np.complex128), _stats

    _stats["splits"] += 1
    w1, _ = sdc_eigvals(B[:r, :r], min_block, leaf_solver, ns_frob, tol,
                        _depth + 1, _stats)
    w2, _ = sdc_eigvals(B[r:, r:], min_block, leaf_solver, ns_frob, tol,
                        _depth + 1, _stats)
    return xp.concatenate([w1, w2]), _stats


def _to_host(A):
    return A if _am(A) is np else A.get()


print("SDC (nonsymmetric) loaded.")


In [ ]:
# =====================================================================
#  11 - Is there even an incumbent? SDC vs whatever the GPU actually offers
# =====================================================================
# The symmetric cells lost to cuSOLVER's syevd, which turned out to be a very
# strong routine: 5.4 gemm-equivalents at n=2048 and falling toward its flop
# floor. This cell asks the same question on the nonsymmetric side, where the
# incumbent may not exist on the device at all.
#
# FIRST it establishes that, rather than assuming it. If cupy.linalg.eig
# raises, then the honest baseline for "I hold A on the GPU and want its
# eigenvalues" is a ROUND TRIP: copy to host, numpy.linalg.eig (LAPACK dgeev),
# copy back -- transfers included, because they are part of what you pay.
#
# Then it races both leaf strategies from cell 10. This is the measurement
# that matters, and neither outcome is predictable from the CPU numbers:
#
#   host-leaf  one split on the device, both halves to dgeev on the CPU. Wins
#              if the transfers are cheap relative to what dgeev saves.
#   deep-leaf  recurse to 2x2, never leave the device. OPTIMIZATION_LOG #25 measured
#              this 4x SLOWER on CPU -- but that verdict assumed a cheap leaf
#              solver existed, which is exactly the assumption this substrate
#              breaks. It also pays ~log2(n) levels of small-block kernel
#              launches, which is the #17 leaf lesson in GPU form.
#
# Accuracy is asserted against dgeev before any time is believed.

import numpy as _np

print("=== does this GPU have a general nonsymmetric eigensolver at all?")
HAVE_GPU_EIG = False
try:
    _t = cp.asarray(_np.random.default_rng(0).standard_normal((64, 64)))
    cp.linalg.eig(_t)
    HAVE_GPU_EIG = True
    print("  cupy.linalg.eig EXISTS -- race SDC against it directly")
except Exception as _e:
    print(f"  no general eig on the device: {type(_e).__name__}: {_e}")
    print("  -> the baseline is a host round trip (transfers included),")
    print("     which is a materially weaker incumbent than syevd was.")


def host_eig_roundtrip(A):
    """The honest baseline when the device has no geev: D2H, dgeev, H2D."""
    return cp.asarray(_np.linalg.eigvals(A.get()))


def matched_err(w, v, nrm):
    w = _np.asarray(w.get() if hasattr(w, "get") else w, dtype=complex)
    v = list(_np.asarray(v, dtype=complex))
    tot = 0.0
    for x in w:
        d = _np.abs(x - _np.array(v)); i = int(_np.argmin(d))
        tot = max(tot, float(d[i])); v.pop(i)
    return tot / nrm


SDC_SIZES = [256, 512, 1024]
print(f"\n  {'n':>6}{'method':>24}{'ms':>10}{'vs host dgeev':>15}"
      f"{'gemm-eq':>9}{'splits':>8}{'dlam':>10}")
for n in SDC_SIZES:
    rng = _np.random.default_rng(2)
    A_h = rng.standard_normal((n, n)) / _np.sqrt(n)
    A = cp.asarray(A_h)
    nrm = float(_np.linalg.norm(A_h, 2))
    wref = _np.linalg.eigvals(A_h)

    t_gemm = timed(lambda: A @ A, cp, reps=5)
    t_host = timed(lambda: host_eig_roundtrip(A), cp, reps=3)
    print(f"\n  n={n}: fp64 gemm {t_gemm*1e3:.2f} ms | host dgeev round trip "
          f"{t_host*1e3:.1f} ms = {t_host/t_gemm:.1f} gemm-equivalents")
    print(f"        (symmetric cuSOLVER syevd measured 5.4-15.3 for scale -- "
          f"a weaker incumbent is the whole opening here)")

    cands = [("host dgeev round trip", lambda: host_eig_roundtrip(A), None),
             ("SDC leaf=host", lambda: sdc_eigvals(A, leaf_solver="host"), None),
             ("SDC leaf=deep", lambda: sdc_eigvals(A, leaf_solver="deep"), None)]
    if HAVE_GPU_EIG:
        cands.insert(1, ("cupy.linalg.eig", lambda: cp.linalg.eig(A)[0], None))

    for label, fn, _ in cands:
        try:
            out = fn()
            w = out[0] if isinstance(out, tuple) else out
            st = out[1] if isinstance(out, tuple) else {"splits": 0}
            sync(cp)
            e = matched_err(w, wref, nrm)
            if not (e < 1e-8):
                print(f"  {n:6d}{label:>24}{'':>10}{'':>15}{'':>9}"
                      f"{st.get('splits', 0):8}{e:10.1e}  NOT ACCURATE")
                continue
            t = timed(fn, cp, reps=2)
            mark = ("   <-- BEATS the host"
                    if (t < t_host and label != "host dgeev round trip") else "")
            print(f"  {n:6d}{label:>24}{t*1e3:10.1f}{t_host/t:14.2f}x"
                  f"{t/t_gemm:9.1f}{st.get('splits', 0):8}{e:10.1e}{mark}")
        except Exception as ex:
            print(f"  {n:6d}{label:>24}  FAILED: {type(ex).__name__}: {ex}")

print("""
  What decides this one

  On the symmetric side the incumbent cost 5.4 gemm-equivalents and nothing
  could fit underneath it. Here the incumbent is a host round trip, and the
  number printed above it is the whole contest: SDC needs ~88, so it wins
  only if the round trip costs MORE than that. Read that line first.

  If leaf=deep beats leaf=host, OPTIMIZATION_LOG #25's leaf verdict has flipped on this
  substrate, and the reason is stated rather than guessable: #25 measured deep
  recursion against a CHEAP leaf solver, and this device has none.

  Transfers are inside the baseline deliberately. Excluding them would measure
  a solver that does not exist -- you cannot get eigenvalues of a device
  matrix without either a device solver or a copy.""")


## What the possible outcomes mean

SSJ costs ~55–80 gemm-equivalents; `dsyevd` costs 8–18 on CPU. For SSJ to win
anywhere, the incumbent's cost in *the same unit* has to rise. Cell 5 prints
exactly that for cuSOLVER ("= N gemm-equivalents"), so you can read the
outcome off that line before even looking at SSJ:

- **cuSOLVER above ~60 gemm-equivalents** — the tridiagonal reduction is
  hurting on this hardware and SSJ is genuinely in range. This is the case the
  method was designed for.
- **cuSOLVER at 15–30** — the gap narrowed but SSJ still loses on flops. The
  interesting configurations are then the ones that cut flops rather than
  reshape them: `mixed`, `+BC32`, and warm starts.
- **cuSOLVER still under ~15** — the all-gemm argument does not pay off here,
  and that is a real finding.

## Reading cell 5 and cell 7 together

These two cells ask different questions and can easily disagree:

- **Cell 5 (cold solve)** is the hardest case for SSJ and the one it is most
  likely to lose. A loss here is expected, not disqualifying.
- **Cell 7 (warm tracking)** is the case SSJ is structurally suited to, and
  the only one where a ratio below 1 is plausible.

If cell 7 shows a win and cell 5 does not, the defensible claim is narrow and
specific: *SSJ is not a general-purpose `syevd` replacement, but it is the
faster option for tracking a slowly-varying matrix.* Resist widening it.

One lever this notebook does not isolate: **mixed precision on tensor cores.**
The fp32 phase does most of the sweeps, and tensor cores run it far faster
than the fp64 units, so the CPU-measured 1.3–1.4× is a floor rather than an
estimate. The `mixed` rows in cell 5 include it, but conflated with everything
else.

## Known weaknesses of this port, stated plainly

- `_orth_ns` syncs once per Newton–Schulz iteration to test its stopping
  criterion (typically 2–5 per sweep). A fixed iteration count would remove
  them at the cost of doing unnecessary work.
- The block pass builds a dense `(keep, keep)` block-diagonal `Qfull` so the
  application is two gemms. That wastes `O(n²)` memory traffic on zeros;
  a batched `matmul` over `(nb, m, m)` views trades that for `nb` small
  kernels. Which wins is a size-dependent question this notebook does not
  settle.
- Sweep counts are hardware-independent, so they are directly comparable to
  the CPU numbers quoted above. Wall times are not comparable across
  machines — only ratios are.

## Cells 9 and 10: the other family, added after the fact

Cells 1–8 test the **SSJ line**. They were written at OPTIMIZATION_LOG #12; the
campaign's champion emerged at #16–23 and is a different family, so cells 9
and 10 were added to close that gap. If you only run part of this notebook,
run cell 10.

**Cell 9** is purification (SP2) + the refinement ladder, ported
backend-agnostically and validated on the NumPy path against the repository's
own implementation across GOE, exact 5-fold ties, 1e-9 clusters and
zero-diagonal matrices, in both precisions — the port reproduces the CPU
numbers to the digit. One tempting GPU optimisation (blending SP2's trace
branch on-device to remove ~30 syncs per projector) was tried and **rejected**:
it silently promotes the fp32 projector to fp64 via a float64 scalar, which
measurably worsened the answer. The rejection is documented at the function
rather than deleted, because the idea is sound and only the dtype was wrong.

**Cell 10** is the measurement the CPU cannot make. On CPU, coarse+ladder does
not pay for exactly one reason: `ssyevd` runs at `dsyevd` *speed* there
(1.01×/0.96×), so the cheap coarse supplier the ladder needs does not exist.
Cell 1 has already told you whether this card breaks that premise. The row to
read is `fp32 eigh + ladder` against `cuSOLVER fp64 (ref)`.

If that row comes in below 1.00×, check the `cuSOLVER fp32 eigh` line printed
just above it first. If fp32 `eigh` is not meaningfully cheaper than fp64
`eigh` on this card, then the *premise* failed rather than the design, and the
result restates the CPU's substrate ambush rather than adding a new one. That
distinction is the difference between "the architecture is wrong" and "this
card is not the substrate for it", and they are not the same finding.


## Cells 10 and 11: the nonsymmetric side

The symmetric race was lost to a strong incumbent. The nonsymmetric one may
not have an incumbent on the device at all — cuSOLVER provides `syevd`,
`syevj`, `sygvd` and the SVDs, but appears to offer no general `geev`. **Cell
11 tests that rather than assuming it**, and the answer changes what the
comparison even means: with no device solver, the honest baseline for "I hold
A on the GPU and want its eigenvalues" is a host round trip, transfers
included.

That matters because of the one number the CPU already established: `dgeev`
costs **131–181 gemm-equivalents** against `dsyevd`'s 17–25. The nonsymmetric
incumbent is ~7× weaker in exactly the unit that decided the symmetric race,
and SDC has op-count *parity* with it (88 vs 89).

Two port changes were forced, and both are worth knowing:

* **The split basis.** CuPy's `qr` has no pivoting, so the pivoted-QR
  extraction of `range(P)` is replaced by the randomized range-finder already
  validated in the purification family — `QR([P G1, (I−P) G2])`, one unpivoted
  QR. Column *order* is load-bearing: OPTIMIZATION_LOG #24 records building the basis
  from a pivoted QR of `[P, I−P]` instead, whose reordering destroys the range
  separation and reported a bogus `‖A21‖ = 2.6e-01` on a symmetric matrix.
* **The leaf.** With no device `geev`, cell 10 offers `leaf_solver="host"`
  (one split, halves to CPU dgeev) and `leaf_solver="deep"` (recurse to 2×2,
  never leave the device). OPTIMIZATION_LOG #25 measured deep recursion **4× slower**
  on CPU — but that verdict assumed a cheap leaf solver existed, which is
  precisely what this substrate removes. If `deep` wins here, that is a
  measured substrate flip, not a tuning result.

Validated on the NumPy path against the repository's `sdc_eigvals` across
Ginibre, planted-real, near-symmetric and companion matrices at n = 128 and
256: both leaf strategies land 7.9e-15 to 1.0e-11, against the repo's
8.7e-15 to 1.9e-13.
